In [0]:
%run ./01-config

In [0]:
# %pip install stomp.py confluent-kafka prometheus_client
# Run this cell once to install dependencies on the cluster
%pip install stomp.py confluent-kafka prometheus_client

In [0]:
import os
import time
import json
import logging
from stomp import Connection, ConnectionListener
from confluent_kafka import Producer
from prometheus_client import Counter, Gauge, start_http_server
from threading import Lock

# Configuration constants
NR_USERNAME = "***************************"
NR_PASSWORD = "*******************" 
CLIENT_ID = "ola-capstone"  # Unique identifier

KAFKA_CONFIG = {
    'bootstrap.servers': 'pkc-12p03.southafricanorth.azure.confluent.cloud:9092',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': '*******************',
    'sasl.password': '*******************************************',
    'linger.ms': 50,
    'batch.num.messages': 1000,
    'compression.type': 'lz4',
    'message.timeout.ms': 30000
}

STOMP_CONFIG = [('publicdatafeeds.networkrail.co.uk', 61618)]
KAFKA_TOPIC = 'train-movements'

# Metrics definitions
MESSAGES_RECEIVED = Counter(
    'stomp_messages_received_total',
    'Total messages received from STOMP',
    ['topic']
)

MESSAGES_PRODUCED = Counter(
    'kafka_messages_produced_total',
    'Messages sent to Kafka',
    ['topic']
)

PROCESSING_TIME = Gauge(
    'message_processing_seconds',
    'Message processing time',
    ['topic']
)

ERROR_COUNTER = Counter(
    'processing_errors_total',  
    'Total processing errors',
    ['topic']
)

# Rate limiting constants
RATE_LIMIT_PER_SECOND = 1  # Adjust this based on API feed limits (e.g., Train Movements: 600/min)
rate_lock = Lock()
last_processed_time = time.time()
processed_count = 0


class EnhancedRailListener(ConnectionListener):
    def __init__(self, producer):
        self.producer = producer
        self._running = True
        self.metrics = {
            'received': MESSAGES_RECEIVED.labels(topic=KAFKA_TOPIC),
            'produced': MESSAGES_PRODUCED.labels(topic=KAFKA_TOPIC),
            'errors': ERROR_COUNTER.labels(topic=KAFKA_TOPIC),
            'processing': PROCESSING_TIME.labels(topic=KAFKA_TOPIC)
        }

    def on_error(self, frame):
        logging.error(f'STOMP Error: {frame.body}')
        self.metrics['errors'].inc()

    def on_message(self, frame):
        global last_processed_time, processed_count

        start_time = time.time()
        self.metrics['received'].inc()

        try:
            # Rate limiting logic
            with rate_lock:
                current_time = time.time()
                elapsed_time = current_time - last_processed_time

                if elapsed_time < 1 and processed_count >= RATE_LIMIT_PER_SECOND:
                    time.sleep(1 - elapsed_time)  # Wait until the next second window

                if elapsed_time >= 1:
                    processed_count = 0
                    last_processed_time = current_time

                processed_count += 1

            message = json.loads(frame.body)
            for event in message:
                self._process_event(event)

            self.producer.poll(0)
            self.metrics['processing'].set(time.time() - start_time)

        except Exception as e:
            self.metrics['errors'].inc()
            logging.exception(f"Failed to process message: {str(e)}")

    def _process_event(self, event):
        try:
            key = event['body']['train_id']
            value = {
                'metadata': event['header'],
                'data': event['body'],
                'processing_ts': int(time.time() * 1000),
                'source': 'network_rail'
            }
            if value['metadata'] and value['metadata']['msg_type'] == '0003':
                self.producer.produce(
                    topic=KAFKA_TOPIC,
                    key=key,
                    value=json.dumps(value['data']),
                    callback=self._delivery_report
                )
                self.metrics['produced'].inc()

        except KeyError as e:
            logging.error(f"Missing key in event: {str(e)}")
            self.metrics['errors'].inc()

    def _delivery_report(self, err, msg):
        if err is not None:
            logging.error(f'Delivery failed: {err}')
            self.metrics['errors'].inc()


def main():
    # Initialize monitoring
    start_http_server(8011)
    logging.basicConfig(level=logging.INFO)

    # Create Kafka producer
    producer = Producer(KAFKA_CONFIG)

    # Configure STOMP connection
    conn = Connection([('publicdatafeeds.networkrail.co.uk', 61618)])
    listener = EnhancedRailListener(producer)

    conn.set_listener('', listener)
    # conn.connect(NR_USERNAME, NR_PASSWORD, wait=True)

    while True:
        try:
            conn.connect(NR_USERNAME, NR_PASSWORD,True,{'heart-beat': '1000000,1000000'})
            conn.subscribe(
                "/topic/TRAIN_MVT_ALL_TOC",
                'kafka-producer',
                'auto'
            )
            while listener._running:
                time.sleep(1)
        except Exception as e:
            logging.error(f"Connection error: {e}")
            time.sleep(5)  # Wait before reconnecting
        finally:
            if conn.is_connected():
                conn.disconnect()



if __name__ == "__main__":
    main()

In [0]:
import json
import requests
from confluent_kafka import Producer
from time import sleep
KAFKA_CONFIG = {
    'bootstrap.servers': 'pkc-12p03.southafricanorth.azure.confluent.cloud:9092',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': '*******************',
    'sasl.password': '**********************************',
    'linger.ms': 50,
    'batch.num.messages': 1000,
    'compression.type': 'lz4',
    'message.timeout.ms': 30000
}
# OpenWeatherMap API configuration
API_KEY = "********************************"

# List of cities to fetch weather data for
CITIES = [
    'Brighton', 
    'Canterbury', 
    'Glasgow', 
    'London', 
    'Cardiff', 
    'Manchester', 
    'Cambridge', 
    'Birmingham', 
    'Newcastle', 
    'Nottingham', 
    'Bristol', 
    'Southampton', 
    'Aylesbury',
    'Liverpool',  # For HILLSIDE, FAZAKERLEY, LIVERPOOL CENTRAL
    'Leeds',      # For HEADINGLEY, HELLIFIELD
    'Sheffield'   # For TINSLEY EAST JN
]

# OpenWeatherMap API URL
URL = 'http://api.openweathermap.org/data/2.5/weather'

# Kafka configuration
KAFKA_BROKER = 'pkc-12p03.southafricanorth.azure.confluent.cloud:9092'
TOPIC = 'weather-updates'

# Kafka Producer Configuration


# Initialize the Kafka Producer
producer = Producer(KAFKA_CONFIG)

def delivery_report(err, msg):
    """Delivery report callback to confirm message delivery."""
    if err is not None:
        print(f"Message delivery failed: {err}")
    else:
        print(f"Message delivered to {msg.topic()} [{msg.partition()}] at offset {msg.offset()}")

def fetch_weather(city):
    """Fetch weather data for a given city using the OpenWeatherMap API."""
    params = {
        'q': city,
        'appid': API_KEY,
        'units': 'metric'
    }
    response = requests.get(URL, params=params)
    
    # Check if the API request was successful
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Failed to fetch weather data for {city}: {response.status_code}, {response.text}")
        return None

def produce_weather_data():
    """Fetch weather data for cities and produce it to Kafka."""
    while True:
        for city in CITIES:
            try:
                # Fetch weather data from the OpenWeatherMap API
                weather_data = fetch_weather(city)
                # print(weather_data)
                if weather_data:
                    # Produce the weather data to the Kafka topic
                    producer.produce(
                        TOPIC,
                        key=city.encode('utf-8'),
                        value=json.dumps(weather_data).encode('utf-8'),
                        callback=delivery_report  # Attach the delivery report callback
                    )
                    producer.flush()  # Ensure messages are sent immediately
                    print(f"Produced weather data for {city}: {weather_data}")
            except Exception as e:
                print(f"Error producing weather data for {city}: {e}")
        
        # Wait 10 minutes before fetching new data
        sleep(600)

if __name__ == "__main__":
    print("Starting Confluent Kafka Weather Data Producer...")
    produce_weather_data()